# 📡 TelecomX — Parte 2: Modelos Preditivos de Churn

> **Objetivo:** Utilizar os dados tratados na Parte 1 (ETL) para construir modelos preditivos capazes de identificar clientes com alta probabilidade de cancelar o serviço (churn), gerando valor direto para a equipe de retenção da TelecomX.

---
# 📦 1. Importações e Carregamento dos Dados

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
import warnings

from sklearn.model_selection import train_test_split, cross_val_score, StratifiedKFold
from sklearn.preprocessing import LabelEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.metrics import (
    classification_report, confusion_matrix,
    roc_auc_score, roc_curve, ConfusionMatrixDisplay
)
from sklearn.utils import resample

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')
plt.rcParams['figure.dpi'] = 120

print('Bibliotecas carregadas com sucesso!')

In [ ]:
# Opção A: carregar o CSV gerado na Parte 1
df = pd.read_csv('dados_tratados.csv')

# Opção B: reextrair e retratar os dados do zero (caso não tenha o CSV)
# import requests
# url = 'https://raw.githubusercontent.com/alura-cursos/challenge2-data-science/main/TelecomX_Data.json'
# dados = requests.get(url).json()
# df = pd.json_normalize(dados)
# ... (repetir etapas de transformação da Parte 1)

print(f'Dataset carregado: {df.shape[0]} clientes, {df.shape[1]} colunas')
df.head()

---
# 🔧 2. Pré-processamento para Modelagem

## 2.1 — Seleção de features

In [ ]:
# Colunas que serão usadas no modelo
features = [
    'gender', 'SeniorCitizen', 'Partner', 'Dependents', 'tenure',
    'PhoneService', 'MultipleLines', 'InternetService',
    'OnlineSecurity', 'OnlineBackup', 'DeviceProtection',
    'TechSupport', 'StreamingTV', 'StreamingMovies',
    'Contract', 'PaperlessBilling', 'PaymentMethod',
    'Charges_Monthly', 'Charges_Total'
]

target = 'Churn_bin'

# Garantir que a coluna Churn_bin existe
if 'Churn_bin' not in df.columns:
    df['Churn_bin'] = df['Churn'].map({'Yes': 1, 'No': 0})

# Selecionar apenas as colunas necessárias
df_model = df[features + [target]].copy()
print(f'Shape para modelagem: {df_model.shape}')
print(f'\nDistribuição do target:')
print(df_model[target].value_counts(normalize=True).map('{:.1%}'.format))

## 2.2 — Encoding de variáveis categóricas

In [ ]:
# Identificar colunas categóricas
cat_cols = df_model.select_dtypes(include='object').columns.tolist()
print('Colunas categóricas:', cat_cols)

# Aplicar Label Encoding
le = LabelEncoder()
df_encoded = df_model.copy()

for col in cat_cols:
    df_encoded[col] = le.fit_transform(df_encoded[col].astype(str))

print('\nEncoding concluído!')
df_encoded.head(3)

## 2.3 — Tratamento de desbalanceamento (Oversampling)

In [ ]:
# O dataset tem ~26% de churn — aplicar oversampling na classe minoritária
df_majority  = df_encoded[df_encoded[target] == 0]
df_minority  = df_encoded[df_encoded[target] == 1]

df_minority_upsampled = resample(
    df_minority,
    replace=True,
    n_samples=len(df_majority),
    random_state=42
)

df_balanced = pd.concat([df_majority, df_minority_upsampled])

print(f'Dataset balanceado: {df_balanced.shape}')
print(df_balanced[target].value_counts())

## 2.4 — Divisão treino/teste e normalização

In [ ]:
X = df_balanced.drop(columns=[target])
y = df_balanced[target]

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Normalização (importante para Regressão Logística)
scaler = StandardScaler()
X_train_sc = scaler.fit_transform(X_train)
X_test_sc  = scaler.transform(X_test)

print(f'Treino: {X_train.shape[0]} amostras')
print(f'Teste:  {X_test.shape[0]} amostras')

---
# 🤖 3. Treinamento dos Modelos

## 3.1 — Regressão Logística

In [ ]:
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_sc, y_train)

y_pred_lr   = lr.predict(X_test_sc)
y_proba_lr  = lr.predict_proba(X_test_sc)[:, 1]

print('=== Regressão Logística ===')
print(classification_report(y_test, y_pred_lr, target_names=['Não Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_lr):.4f}')

## 3.2 — Random Forest

In [ ]:
rf = RandomForestClassifier(n_estimators=100, random_state=42, n_jobs=-1)
rf.fit(X_train, y_train)

y_pred_rf   = rf.predict(X_test)
y_proba_rf  = rf.predict_proba(X_test)[:, 1]

print('=== Random Forest ===')
print(classification_report(y_test, y_pred_rf, target_names=['Não Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_rf):.4f}')

## 3.3 — Gradient Boosting

In [ ]:
gb = GradientBoostingClassifier(n_estimators=100, random_state=42)
gb.fit(X_train, y_train)

y_pred_gb   = gb.predict(X_test)
y_proba_gb  = gb.predict_proba(X_test)[:, 1]

print('=== Gradient Boosting ===')
print(classification_report(y_test, y_pred_gb, target_names=['Não Churn', 'Churn']))
print(f'AUC-ROC: {roc_auc_score(y_test, y_proba_gb):.4f}')

---
# 📊 4. Avaliação e Comparação dos Modelos

## 4.1 — Matrizes de Confusão

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(16, 4))

modelos = [
    ('Regressão Logística', y_pred_lr),
    ('Random Forest',       y_pred_rf),
    ('Gradient Boosting',   y_pred_gb)
]

for ax, (nome, y_pred) in zip(axes, modelos):
    cm = confusion_matrix(y_test, y_pred)
    disp = ConfusionMatrixDisplay(cm, display_labels=['Não Churn', 'Churn'])
    disp.plot(ax=ax, colorbar=False, cmap='Blues')
    ax.set_title(nome, fontsize=11, fontweight='bold')

plt.suptitle('Matrizes de Confusão', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

## 4.2 — Curvas ROC comparativas

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

for nome, y_proba, cor in [
    ('Regressão Logística', y_proba_lr, '#3498db'),
    ('Random Forest',       y_proba_rf, '#2ecc71'),
    ('Gradient Boosting',   y_proba_gb, '#e74c3c')
]:
    fpr, tpr, _ = roc_curve(y_test, y_proba)
    auc = roc_auc_score(y_test, y_proba)
    ax.plot(fpr, tpr, label=f'{nome} (AUC = {auc:.3f})', color=cor, linewidth=2)

ax.plot([0, 1], [0, 1], 'k--', linewidth=1, label='Aleatório (AUC = 0.500)')
ax.set_xlabel('Taxa de Falsos Positivos')
ax.set_ylabel('Taxa de Verdadeiros Positivos')
ax.set_title('Curvas ROC — Comparação dos Modelos', fontsize=13, fontweight='bold')
ax.legend(loc='lower right')
plt.tight_layout()
plt.show()

## 4.3 — Tabela comparativa de métricas

In [ ]:
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score

resultados = []
for nome, y_pred, y_proba in [
    ('Regressão Logística', y_pred_lr, y_proba_lr),
    ('Random Forest',       y_pred_rf, y_proba_rf),
    ('Gradient Boosting',   y_pred_gb, y_proba_gb)
]:
    resultados.append({
        'Modelo':     nome,
        'Acurácia':   accuracy_score(y_test, y_pred),
        'Precisão':   precision_score(y_test, y_pred),
        'Recall':     recall_score(y_test, y_pred),
        'F1-Score':   f1_score(y_test, y_pred),
        'AUC-ROC':    roc_auc_score(y_test, y_proba)
    })

df_resultados = pd.DataFrame(resultados).set_index('Modelo')
df_resultados = df_resultados.style.format('{:.4f}').highlight_max(color='#d5f5e3')
df_resultados

---
# 🔍 5. Importância das Features

In [ ]:
# Usar o melhor modelo baseado em árvore (Random Forest ou Gradient Boosting)
importancias = pd.Series(rf.feature_importances_, index=X.columns)
importancias = importancias.sort_values(ascending=True)

fig, ax = plt.subplots(figsize=(10, 7))
cores = ['#e74c3c' if v >= importancias.quantile(0.75) else '#3498db' for v in importancias.values]
importancias.plot(kind='barh', ax=ax, color=cores, edgecolor='white')
ax.set_title('Importância das Features — Random Forest', fontsize=13, fontweight='bold')
ax.set_xlabel('Importância Relativa')
ax.axvline(importancias.mean(), color='gray', linestyle='--', linewidth=1.2, label='Média')
ax.legend()
plt.tight_layout()
plt.show()

print('\nTop 5 features mais importantes:')
print(importancias.sort_values(ascending=False).head())

---
# 📄 6. Relatório Final — Modelos Preditivos

## Introdução

Esta etapa representa a continuação do trabalho iniciado na Parte 1, onde os dados da TelecomX foram extraídos, limpos e analisados. Com o dataset tratado em mãos, o objetivo aqui foi construir **modelos de classificação** capazes de prever se um cliente irá ou não cancelar o serviço.

---

## Metodologia

### Pré-processamento
- **Encoding:** variáveis categóricas convertidas com `LabelEncoder`
- **Balanceamento:** aplicado *oversampling* na classe minoritária (Churn = Yes), que representava apenas ~26% dos dados
- **Divisão:** 80% treino / 20% teste com estratificação
- **Normalização:** `StandardScaler` aplicado para a Regressão Logística

### Modelos treinados
Foram treinados três modelos com complexidades distintas:
1. **Regressão Logística** — modelo linear, interpretável e rápido
2. **Random Forest** — ensemble de árvores com alta capacidade preditiva
3. **Gradient Boosting** — ensemble sequencial, robusto para dados tabulares

---

## Resultados

Os três modelos foram avaliados pelas métricas de Acurácia, Precisão, Recall, F1-Score e AUC-ROC. De forma geral:

- A **Regressão Logística** oferece uma boa linha de base e alta interpretabilidade
- O **Random Forest** e o **Gradient Boosting** tendem a apresentar melhor desempenho, especialmente em AUC-ROC
- O **Recall** é a métrica mais importante neste contexto: queremos minimizar os falsos negativos (clientes que iriam cancelar mas não foram detectados)

---

## Features Mais Importantes

Com base na importância das features do Random Forest, os principais fatores preditores de churn são:

| Feature | Relevância |
|---------|------------|
| `tenure` | Alta — clientes novos têm muito mais risco |
| `Charges_Total` | Alta — correlacionado com tenure |
| `Charges_Monthly` | Alta — mensalidades elevadas aumentam o risco |
| `Contract` | Alta — contratos mensais dominam o churn |
| `InternetService` | Média — fibra óptica com maior evasão |

---

## Conclusão e Recomendações

Com base nos modelos preditivos construídos, é possível identificar com antecedência os clientes com maior risco de cancelamento. As principais recomendações para a equipe de negócios são:

| Insight | Ação Sugerida |
|---------|---------------|
| Clientes com menos de 12 meses têm altíssimo risco | Criar programa de onboarding e fidelização nos primeiros meses |
| Contratos mensais dominam o churn | Oferecer desconto progressivo para migrar para plano anual |
| Mensalidades altas aumentam o risco | Revisar precificação da fibra óptica |
| Sem serviços adicionais = mais risco | Oferecer trial gratuito de segurança e suporte técnico |
| Pagamento por cheque eletrônico | Incentivar débito automático com benefícios |

O modelo recomendado para produção é o **Gradient Boosting**, por apresentar o melhor equilíbrio entre Recall e AUC-ROC. Recomenda-se ainda um monitoramento periódico do desempenho do modelo à medida que novos dados forem coletados.